In [49]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [50]:
from broharness.llms.bedrock import bedrock, UserMessage, AIMessage, SystemMessage
from broharness.flows.skill_call import SkillCall
from broharness.flows.tool_call import ToolCall
from broharness.flows.tool_use import ToolUse
from broharness.flows.ask_user_question import AskUserQuestion
from broharness.flows.fail_recovery import FailRecovery
from broharness.flows.answer import Answer
from broflow import BaseTask, TaskRegistry, Flow
from pathlib import Path
import yaml
import sys
import subprocess
from broskill import SkillControl, ToolControl
from functools import partial
from broharness.toolblock import (
    tool_to_yaml, 
    load_skill_tool, 
    load_skill_extension_tool, 
    load_tool_tool, 
    ask_user_question_tool
)
from broharness.codeblock import parse_json_codeblock
from broharness.data_model import Process, State, LLMUse


ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
sc.list_skills()
tc = ToolControl(sc)

In [51]:
from broskill.processing.tool import to_args

In [52]:
tc.load_tool('read-file', 'scripts/read_file.py')

Tool(name='read_file', description="Read exactly one file's content, found via a glob pattern that must match a single file.", args=[Arg(name='pattern', type='string', description="Glob pattern, relative to the project root, that matches exactly one file (e.g. 'skills/tell-joke/references/dad-joke.md').", required=True)], path=WindowsPath('D:/study-on-agent/skills/read-file/scripts/read_file.py'))

In [53]:
TOOLS = dict(
    load_skill=sc.load_skill,
    load_skill_extension=sc.load_skill_extension,
    load_tool=tc.load_tool,
)

In [54]:
skill_call = SkillCall(name='skill-call', llm=bedrock, system_prompt='')
tool_call = ToolCall(name='tool-call', llm=bedrock, system_prompt='')
tool_use = ToolUse(name='tool-use', llm=bedrock, system_prompt='')
ask_user_question = AskUserQuestion(name='ask-user-question', llm=bedrock, system_prompt='')
fail_recovery = FailRecovery(name='fail-recovery', llm=bedrock, system_prompt='')
answer = Answer(name='answer', llm=bedrock, system_prompt='')

In [55]:
registry = TaskRegistry()
registry.register(Process.SKILL_CALL, skill_call)
registry.register(Process.TOOL_CALL, tool_call)
registry.register(Process.TOOL_USE, tool_use)
registry.register(Process.ASK_USER_QUESTION, ask_user_question)
registry.register(Process.FAIL_RECOVERY, fail_recovery)
registry.register(Process.ANSWER, answer)

flow = Flow(registry)

In [56]:
system_prompt = "You're Andy who is the best bro in the world. Always response in bro-tone with chill and mellow manner."

In [57]:
# this trigger load_skill with skill_name='tell-jokes'
content = "tell me some jokes."
# this trigger ask_user_question
# content = "What's the capital of France?"
# this trigger nothing
# content = "1+1 is?"
messages = [UserMessage(content)]
state = State(
    root=ROOT,
    skill_dir=SKILL_DIR,
    messages=messages,
    session_messages=messages.copy(),
    system_prompt=system_prompt,
    skill_control=sc,
    tool_control=tc,
    tools=TOOLS,
    session_tools=TOOLS.copy(),
    debug=messages.copy()
)

_ = flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)

D:\study-on-agent\src\broharness\flows\skill_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill', 'input': {'skill_name': 'tell-joke'}}]
load_skill passed
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'ask_user_question', 'input': {'question': 'What kind of joke would you like to hear?'}}]
ask_user_question passed
D:\study-on-agent\src\broharness\flows\ask_user_question.py
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill_extension', 'input': {'skill_name': 'tell-joke', 'path': 'references/dad-joke.md'}}, {'name': 'load_skill_extension', 'input': {'skill_name': 'tell-joke', 'path': 'references/riddle.md'}}]
load_skill_extension passed
load_skill_extension passed
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\answer.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'na

In [58]:
state.messages

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Why was the school bus always late? Because it kept stopping to pick up more knowledge!'}]}]

In [59]:
state.session_messages

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': 'What kind of joke would you like to hear?'}]},
 {'role': 'user', 'content': [{'text': 'mix of dad jokes and riddle.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Okay, great! And what topic would you like the jokes to be about?'}]},
 {'role': 'user', 'content': [{'text': 'School bus.'}]},
 {'role': 'assistant',
  'content': [{'text': 'Why was the school bus always late? Because it kept stopping to pick up more knowledge!'}]}]

In [60]:
state.registered_skills

{'tell-joke': '# Tell Joke\n\n## Instructions\n\n- Ask the user which kind of joke they\'d like: dad jokes, puns, knock-knock\n  jokes, one-liners, riddles, or a mix of any of these.\n- Also ask what topic the joke should be about (e.g. work, food, animals) -- as its\n  own separate `ask_user_question` call, not bundled into the joke-type question.\n  No topic preference is a fine answer too; don\'t force a pick.\n- If they already said the type and/or topic in their request, don\'t ask again for\n  whichever part they already gave.\n- If their answer is unclear, ask a short clarifying question. Call `ask_user_question`.\n- Tell exactly one joke at a time, in your own words. Don\'t paste a reference file\'s\n  contents back verbatim -- use it as a style guide to write a fresh joke on the\n  requested topic, not a script to copy.\n- If more than one reference is loaded (a "both"/"all"/multi-type request), that\'s\n  still one joke, not one per style -- fuse every loaded style\'s charact

In [61]:
state.extension_skills

{'tell-joke': '# Dad jokes\n\nDad jokes are short, deadpan one-liners built on the most literal, groan-worthy\nreading of a common phrase or word. The humor comes from how obvious and corny\nthe pun is -- being "so bad it\'s good" is the point, not a flaw.\n\n## Characteristics\n\n- Plays on a double meaning or overly literal reading of an everyday phrase or idiom.\n- Delivered deadpan, straight-faced -- no self-aware "get it?" follow-up.\n- Family-friendly, no edge or innuendo.\n- Short: one or two sentences, the setup and punchline folded together.\n- Usually built around a mundane, everyday topic (food, chores, animals, objects,\n  work) -- swap in whatever topic the user asked for.\n\nUse this as a style guide to write a fresh, original dad joke on the requested\ntopic -- not a script to copy from verbatim.\n# Riddle jokes\n\nA riddle joke poses a question and lets the listener guess before revealing an\nanswer that\'s clever, absurd, or punny rather than literally correct.\n\n## C

In [62]:
state.session_tools

{'load_skill': <bound method SkillControl.load_skill of <broskill.processing.skill.SkillControl object at 0x00000267CBB93C80>>,
 'load_skill_extension': <bound method SkillControl.load_skill_extension of <broskill.processing.skill.SkillControl object at 0x00000267CBB93C80>>,
 'load_tool': <bound method ToolControl.load_tool of <broskill.processing.tool.ToolControl object at 0x00000267CCD727B0>>}

In [63]:
state.error_message

''

In [64]:
state.debug

[{'role': 'user', 'content': [{'text': 'tell me some jokes.'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill", "input": { "skill_name": "tell-joke" } }\n  ]\n}\n```'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "ask_user_question",\n      "input": {\n        "question": "What kind of joke would you like to hear?"\n      }\n    }\n  ]\n}\n```'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill_extension", "input": { "skill_name": "tell-joke", "path": "references/dad-joke.md" } },\n    { "name": "load_skill_extension", "input": { "skill_name": "tell-joke", "path": "references/riddle.md" } }\n  ]\n}\n```'}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    { "name": "load_skill_extension", "input": { "skill_name": "tell-joke", "path": "references/dad-joke.md" } },\n    { "name": "load_skill_